# Ordinal Regression CNN

## Libraries

In [24]:
import time
import numpy as np
import pandas as pd
import os
import torch.nn as nn
import torch.nn.functional as F
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image

In [25]:
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

## Downloading the Dataset

In [26]:
!git clone https://github.com/afad-dataset/tarball-lite.git

Cloning into 'tarball-lite'...
remote: Enumerating objects: 37, done.
remote: Total 37 (delta 0), reused 0 (delta 0), pack-reused 37 (from 1)
Receiving objects: 100% (37/37), 865.47 MiB | 60.45 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (30/30), done.


In [27]:
!cat tarball-lite/AFAD-Lite.tar.xz* > tarball-lite/AFAD-Lite.tar.xz

In [28]:
!tar xf tarball-lite/AFAD-Lite.tar.xz

In [29]:
rootDir = 'AFAD-Lite'

files = [os.path.relpath(os.path.join(dirpath, file), rootDir)
         for (dirpath, dirnames, filenames) in os.walk(rootDir) 
         for file in filenames if file.endswith('.jpg')]

In [30]:
len(files)

59344

In [31]:
d = {}

d['age'] = []
d['gender'] = []
d['file'] = []
d['path'] = []

for f in files:
    age, gender, fname = f.split('/')
    if gender == '111':
        gender = 'male'
    else:
        gender = 'female'
        
    d['age'].append(age)
    d['gender'].append(gender)
    d['file'].append(fname)
    d['path'].append(f)

In [32]:
df = pd.DataFrame.from_dict(d)
df.head()

,age,gender,file,path
0,37,female,477828-1.jpg,37/112/477828-1.jpg
1,37,female,124327-0.jpg,37/112/124327-0.jpg
2,37,female,407220-1.jpg,37/112/407220-1.jpg
3,37,female,410403-0.jpg,37/112/410403-0.jpg
4,37,female,120435-0.jpg,37/112/120435-0.jpg


In [33]:
df['age'].min()

'18'

In [34]:
df['age'] = df['age'].values.astype(int) - 18

In [35]:
np.random.seed(123)
msk = np.random.rand(len(df)) < 0.8
df_train = df[msk]
df_test = df[~msk]

In [36]:
df_train.set_index('file', inplace=True)
df_train.to_csv('training_set_lite.csv')

In [37]:
df_test.set_index('file', inplace=True)
df_test.to_csv('test_set_lite.csv')

In [38]:
num_ages = np.unique(df['age'].values).shape[0]
print(num_ages)

22


## Settings

In [39]:
# Device
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

NUM_WORKERS = 4

NUM_CLASSES = 22
BATCH_SIZE = 512
NUM_EPOCHS = 10
LEARNING_RATE = 0.0005
RANDOM_SEED = 123
GRAYSCALE = False

TRAIN_CSV_PATH = 'training_set_lite.csv'
TEST_CSV_PATH = 'test_set_lite.csv'
IMAGE_PATH = 'AFAD-Lite'

## Dataset Loaders

In [40]:
class AFADDatasetAge(Dataset):
    """Custom Dataset for loading AFAD face images"""

    def __init__(self, csv_path, img_dir, transform=None):

        df = pd.read_csv(csv_path, index_col=0)
        self.img_dir = img_dir
        self.csv_path = csv_path
        self.img_paths = df['path']
        self.y = df['age'].values
        self.transform = transform

    def __getitem__(self, index):
        img = Image.open(os.path.join(self.img_dir,
                                      self.img_paths[index]))

        if self.transform is not None:
            img = self.transform(img)

        label = self.y[index]

        return img, label

    def __len__(self):
        return self.y.shape[0]

In [41]:
custom_transform = transforms.Compose([transforms.Resize((128, 128)),
                                       transforms.RandomCrop((120, 120)),
                                       transforms.ToTensor()])

train_dataset = AFADDatasetAge(csv_path=TRAIN_CSV_PATH,
                               img_dir=IMAGE_PATH,
                               transform=custom_transform)


custom_transform2 = transforms.Compose([transforms.Resize((128, 128)),
                                        transforms.CenterCrop((120, 120)),
                                        transforms.ToTensor()])

test_dataset = AFADDatasetAge(csv_path=TEST_CSV_PATH,
                              img_dir=IMAGE_PATH,
                              transform=custom_transform2)


train_loader = DataLoader(dataset=train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=NUM_WORKERS)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=NUM_WORKERS)

## Model

In [42]:
def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class ResNet(nn.Module):

    def __init__(self, block, layers, num_classes, grayscale):
        self.num_classes = num_classes
        self.inplanes = 64
        if grayscale:
            in_dim = 1
        else:
            in_dim = 3
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv2d(in_dim, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AvgPool2d(7, stride=1, padding=2)
        self.fc = nn.Linear(2048 * block.expansion, num_classes)
        self.a = torch.nn.Parameter(torch.zeros(
            self.num_classes).float().normal_(0.0, 0.1).view(-1, 1))

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, (2. / n)**.5)
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        logits = self.fc(x)
        probas = torch.softmax(logits, dim=1)
        predictions = ((self.num_classes-1)
                       * torch.sigmoid(probas.mm(self.a).view(-1)))
        return logits, probas, predictions


def resnet34(num_classes, grayscale):
    """Constructs a ResNet-34 model."""
    model = ResNet(block=BasicBlock,
                   layers=[3, 4, 6, 3],
                   num_classes=num_classes,
                   grayscale=grayscale)
    return model

In [43]:
def cost_fn(targets, predictions):
    return torch.mean((targets.float() - predictions)**2)


torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
model = resnet34(NUM_CLASSES, GRAYSCALE)

model.to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Training

In [48]:
def compute_mae_and_mse(model, data_loader):
    mae, mse, num_examples = torch.tensor([0.]), torch.tensor([0.]), 0
    for features, targets in data_loader:
        features = features.to(DEVICE)
        targets = targets.float().to(DEVICE)
        logits, probas, predictions = model(features)
        assert len(targets.size()) == 1
        assert len(predictions.size()) == 1
        predicted_labels = torch.round(predictions).float()
        num_examples += targets.size(0)
        mae += torch.abs(predicted_labels - targets).sum()
        mse += torch.sum((predicted_labels - targets)**2)
    mae = mae / num_examples
    mse = mse / num_examples
    return mae, mse


start_time = time.time()
for epoch in range(NUM_EPOCHS):

    model.train()
    for batch_idx, (features, targets) in enumerate(train_loader):

        features = features.to(DEVICE)
        targets = targets.to(DEVICE)

        # FORWARD AND BACK PROP
        logits, probas, predictions = model(features)
        assert len(targets.size()) == 1
        assert len(predictions.size()) == 1
        cost = cost_fn(targets, predictions)
        optimizer.zero_grad()

        cost.backward()

        # UPDATE MODEL PARAMETERS
        optimizer.step()

        # LOGGING
        if not batch_idx % 150:
            s = ('Epoch: %03d/%03d | Batch %04d/%04d | Cost: %.4f'
                 % (epoch+1, NUM_EPOCHS, batch_idx,
                     len(train_dataset)//BATCH_SIZE, cost))
            print(s)

    s = 'Time elapsed: %.2f min' % ((time.time() - start_time)/60)
    print(s)

/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a f

Epoch: 001/010 | Batch 0000/0092 | Cost: 28.2625
Time elapsed: 0.74 min


/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a f

Epoch: 002/010 | Batch 0000/0092 | Cost: 29.2770
Time elapsed: 1.47 min


/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a f

...

Epoch: 008/010 | Batch 0000/0092 | Cost: 20.4625
Time elapsed: 5.89 min


/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a f

Epoch: 009/010 | Batch 0000/0092 | Cost: 19.7164
Time elapsed: 6.62 min


/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.img_paths[index]))
/tmp/ipykernel_36/1400844432.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a f

Epoch: 010/010 | Batch 0000/0092 | Cost: 21.7963
Time elapsed: 7.36 min
